# 7. The one-at-a-time sweep (FM-17)

`config/sweeps/oat_baseline.json` lists the variables; each takes each planned value from
`config/model.json` alone, plus the named 'roof' region. Run from the terminal:

`nohup python -m efov sweep config/sweeps/oat_baseline.json --out runs/oat --np 24 > runs/oat.log 2>&1 &`

The baseline runs first and fixes the gain for every variant (FM-19). Variants that change only
electrodes, noise or analysis reuse the baseline simulation; the others compact their raw output after
their report (FM-20). Failed variants are recorded and never pooled.

In [ ]:
import os, sys
from pathlib import Path
REPOSITORY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPOSITORY))

# ---- settings: edit here, or set EFOV_CONFIG / EFOV_OUT / EFOV_NP ----
CONFIG = os.environ.get("EFOV_CONFIG", str(REPOSITORY / "config" / "model.json"))
OUT    = os.environ.get("EFOV_OUT",    str(REPOSITORY / "runs" / "v4_reference"))
NP     = int(os.environ.get("EFOV_NP", "24"))      # MPI processes for openCARP
# openCARP's own MPI launcher (efov-runner image); needed when NP > 1
os.environ.setdefault("MPIEXEC_BIN", "/usr/local/lib/opencarp/lib/petsc/bin/mpiexec")

import json
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from efov.config import loadConfig
from efov.pipeline import Run

config = loadConfig(CONFIG)
run = Run(config, OUT, processCount=NP)
print(f"{config['modelName']} -> {run.out}")
run.status()

In [ ]:
SWEEP_OUT = REPOSITORY / "runs" / "oat"
status = json.loads((SWEEP_OUT / "sweepStatus.json").read_text()) if (SWEEP_OUT / "sweepStatus.json").exists() else {}
pd.DataFrame({k: dict(status=v["status"], overrides=v.get("overrides")) for k, v in status.items()}).T

In [ ]:
# Pool the inferior baseline and the roof variant for Wang's regional comparison:
# subprocess.run([sys.executable, "-m", "efov", "wang-combine", str(SWEEP_OUT / "variants" / "baseline"),
#                 str(SWEEP_OUT / "variants" / "roof"), "--out", str(SWEEP_OUT / "wang_regions")], cwd=REPOSITORY)